# 5.2 融合案例：用Cube计算矩阵乘，用Vector加偏置

## 概述

4.2节已经介绍了分块矩阵乘和K方向累加。本节在它后面接上一段向量加法，计算 `Y = A @ B.T + Bias`。Cube计算矩阵乘，Vector给结果加偏置，二者通过中间结果的传递组成一个融合算子。

本节围绕一份完整的TileLang代码展开：每段代码读写什么数据、在什么位置计算，以及为什么这样写。

### 学习目标

- 读懂输入输出声明、任务分块和局部缓冲区分配。
- 将数据搬运、Cube计算和Vector计算对应到具体DSL语句。
- 解释矩阵乘结果怎样交给Vector，以及各部分结果怎样写回正确的位置。

<small>线上环境没有昇腾950，可直接阅读本页代码和实测结果；自行运行请按1.4准备环境，并以本章目录为工作目录。Notebook用户从下方准备单元开始顺序执行。</small>

<small>下方辅助代码用于创建目录和启动独立进程，阅读时可先看第1部分。</small>

In [1]:
from pathlib import Path
import re
import subprocess
import sys

Path("src").mkdir(exist_ok=True)
Path("answer").mkdir(exist_ok=True)

def run_and_show(command):
    # NPU程序在独立进程中运行；当前Notebook内核不导入torch_npu。
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    # 只省略安装目录；数值、校验结果和报错内容保持原样。
    output = result.stdout.replace(str(Path.cwd()), ".")
    output = re.sub(r"/home/[^\s]+", "<已省略安装路径>", output)
    print(output, end="")
    result.check_returncode()
    return result


## 1. 先看完整的算子代码

本节沿用4.2的分块与K方向累加方法。为便于看清Cube与Vector之间的数据传递，这里使用较小的矩阵，并让**每个逻辑任务只处理一个输出块**；第4章中一个任务循环处理多个输出块的写法，这里先不采用。

本例固定使用以下参数，后文都以这组参数解释代码：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">对象或参数</th>
<th style="text-align: left; vertical-align: top;">取值</th>
<th style="text-align: left; vertical-align: top;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>A</code></td>
<td style="text-align: left; vertical-align: top;">(128,128)，float16</td>
<td style="text-align: left; vertical-align: top;">输入矩阵，形状为(M,K)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>B</code></td>
<td style="text-align: left; vertical-align: top;">(128,128)，float16</td>
<td style="text-align: left; vertical-align: top;">权重矩阵，按(N,K)存放，计算时使用B.T</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>Bias</code></td>
<td style="text-align: left; vertical-align: top;">(128,)，float32</td>
<td style="text-align: left; vertical-align: top;">每个输出列对应一个偏置</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>Y</code></td>
<td style="text-align: left; vertical-align: top;">(128,128)，float32</td>
<td style="text-align: left; vertical-align: top;">最终结果，形状为(M,N)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">BM、BN、BK</td>
<td style="text-align: left; vertical-align: top;">都为64</td>
<td style="text-align: left; vertical-align: top;">一个结果块有64行64列，每轮处理K方向的64个元素</td>
</tr>
</tbody>
</table>

偏置按列使用，即 `Y[r,c] = (A @ B.T)[r,c] + Bias[c]`。同一列的不同行加上相同的偏置。

下面是完整的kernel定义。先找到三段主干：K循环中的矩阵乘、`T.SimdVF()`中的偏置相加，以及最后的写回。其余语句会在后面逐段解释。

**写入kernel定义**

<small>写入提示：本单元只保存代码。<code>Overwriting src/matmul_bias.py</code>表示覆盖写入成功；第6部分将追加Host代码并执行算子。</small>

In [2]:
%%writefile src/matmul_bias.py
import tilelang
import tilelang.ascend.language as T
from tilelang.ascend.language import simd as S

tilelang.disable_cache()

M = N = K = 128
BM = BN = BK = 64

@T.prim_func
def matmul_bias(
    A: T.Tensor((M, K), "float16"),
    B: T.Tensor((N, K), "float16"),
    Bias: T.Tensor((N,), "float32"),
    Y: T.Tensor((M, N), "float32"),
):
    with T.Kernel((M // BM) * (N // BN)) as bx:
        row = bx // (N // BN)
        col = bx % (N // BN)

        a = T.alloc_l1((BM, BK), "float16")
        b = T.alloc_l1((BN, BK), "float16")
        acc = T.alloc_l0c((BM, BN), "float32")
        v = T.alloc_shared((BM // 2, BN), "float32")
        bias = T.alloc_shared((BN,), "float32")

        for ki in T.serial(K // BK):
            T.copy(A[row * BM:(row + 1) * BM,
                     ki * BK:(ki + 1) * BK], a)
            T.copy(B[col * BN:(col + 1) * BN,
                     ki * BK:(ki + 1) * BK], b)
            T.gemm(a, b, acc,
                   transpose_B=True, clear_accum=(ki == 0))

        T.dual_copy(acc, v)
        T.copy(Bias[col * BN:(col + 1) * BN], bias)

        with T.SimdVF():
            full = S.pset(32, "PAT_ALL")
            for i in T.serial(BM // 2):
                value = S.vld(v[i, 0])
                bv = S.vld(bias[0])
                result = S.vadd(value, bv, full)
                S.vsts(v[i, 0], result, full)

        T.dual_copy(v, Y[row * BM:(row + 1) * BM,
                         col * BN:(col + 1) * BN])


Overwriting src/matmul_bias.py


三条导入语句分别提供编译接口`tilelang`、以`T`为前缀的TileLang语言接口，以及以`S`为前缀的SIMD指令。这里的代码用于描述NPU上执行的计算。`@T.prim_func`将带类型注解的函数定义转换为TileLang的kernel描述；写出这个定义，还没有启动NPU计算。

## 2. 声明输入输出，并选定当前结果块

### 参数注解说明张量是什么

```python
A: T.Tensor((M, K), "float16")
B: T.Tensor((N, K), "float16")
Bias: T.Tensor((N,), "float32")
Y: T.Tensor((M, N), "float32")
```

这四行是函数参数注解，告诉编译器每个张量的形状和元素类型。A、B、Bias是输入，Y用于接收结果，它们的数据存放在GM中。GM指设备的全局内存，可供参与计算的C核和V核访问。

`T.Tensor`本身不会复制数据，也不会给张量填值。实际数据由调用方提供；这里先约定接口。

### 用任务编号计算块的位置

```python
with T.Kernel((M // BM) * (N // BN)) as bx:
    row = bx // (N // BN)
    col = bx % (N // BN)
```

128×128的输出按64×64分块，共有2行、2列，合计4个块。`T.Kernel(4)`为本例建立4份逻辑任务，每份处理一个输出块，`bx`是当前任务的编号，取0、1、2、3。

每行有`N // BN = 2`个块。任务编号除以2的整数商给出行块编号，余数给出列块编号：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;"><code>bx</code></th>
<th style="text-align: left; vertical-align: top;"><code>row</code></th>
<th style="text-align: left; vertical-align: top;"><code>col</code></th>
<th style="text-align: left; vertical-align: top;">当前任务负责的输出区域</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;"><code>Y[0:64,0:64]</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;"><code>Y[0:64,64:128]</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;"><code>Y[64:128,0:64]</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">3</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;"><code>Y[64:128,64:128]</code></td>
</tr>
</tbody>
</table>

![由逻辑任务编号找到输出块](images/05_02_bias_tiles.png)

<small>图5-1　任务编号与输出块的位置</small>

图中橙色区域对应bx=1：它在第0行块、第1列块，所以起始行是0×64，起始列是1×64。`row`和`col`是块编号，乘以BM、BN后才得到元素的起始下标。切片[0:64]包含下标0到63。

**任务与物理核：** `bx`用于选定要处理的数据，**不等于物理核编号**。编译器根据kernel中的矩阵计算和向量计算组织C/V执行。

## 3. 为Cube和Vector准备局部缓冲区

```python
a = T.alloc_l1((BM, BK), "float16")
b = T.alloc_l1((BN, BK), "float16")
acc = T.alloc_l0c((BM, BN), "float32")
v = T.alloc_shared((BM // 2, BN), "float32")
bias = T.alloc_shared((BN,), "float32")
```

这五行分配计算时使用的局部缓冲区。分配只确定存储位置、形状和类型，数据要由后面的搬运或计算语句写入。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">变量</th>
<th style="text-align: left; vertical-align: top;">这行代码分配了什么</th>
<th style="text-align: left; vertical-align: top;">用来保存什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>a</code></td>
<td style="text-align: left; vertical-align: top;">L1中的64×64个float16元素</td>
<td style="text-align: left; vertical-align: top;">A在当前K段的输入块</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>b</code></td>
<td style="text-align: left; vertical-align: top;">L1中的64×64个float16元素</td>
<td style="text-align: left; vertical-align: top;">B在当前K段的输入块</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>acc</code></td>
<td style="text-align: left; vertical-align: top;">L0C中的64×64个float32元素</td>
<td style="text-align: left; vertical-align: top;">Cube计算得到的累加结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>v</code></td>
<td style="text-align: left; vertical-align: top;">每个V核UB中的32×64个float32元素</td>
<td style="text-align: left; vertical-align: top;">交给该V核处理的半个结果块</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>bias</code></td>
<td style="text-align: left; vertical-align: top;">每个V核UB中的64个float32元素</td>
<td style="text-align: left; vertical-align: top;">当前输出块对应的64个列偏置</td>
</tr>
</tbody>
</table>

`alloc_l1`选择Cube侧的L1存储，`alloc_l0c`选择矩阵乘的累加存储L0C。在当前昇腾后端中，`alloc_shared`用于这里的UB缓冲区，供Vector读写；它不是GM，也不表示所有物理核共同使用同一份缓冲区。

**为什么`acc`有64行，`v`只有32行？**

本例采用的C/V组合让两个V核各处理一半行：一个处理前32行，另一个处理后32行。每个V核有自己的`v`。两份32×64合起来，正好覆盖Cube得到的64×64结果。稍后的`T.dual_copy`负责这次传递。

大写A、B、Bias是GM中的完整输入；小写a、b、bias是当前计算使用的局部数据。区分这两组名字，可以看清每次搬运的来源和去向。

## 4. 逐轮搬入数据，用Cube计算并累加

```python
for ki in T.serial(K // BK):
    T.copy(A[row * BM:(row + 1) * BM, ki * BK:(ki + 1) * BK], a)
    T.copy(B[col * BN:(col + 1) * BN, ki * BK:(ki + 1) * BK], b)
    T.gemm(a, b, acc, transpose_B=True, clear_accum=(ki == 0))
```

`T.serial(K // BK)`表示按ki的顺序执行循环。本例K=128、BK=64，因此ki依次为0、1。两轮处理的是**同一个结果块在K方向上的两部分计算**。

继续取bx=1，即row=0、col=1。将这些数代入两条`T.copy`，就能得到实际读取的区域：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">ki</th>
<th style="text-align: left; vertical-align: top;">从A搬到a的数据</th>
<th style="text-align: left; vertical-align: top;">从B搬到b的数据</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;"><code>A[0:64,0:64]</code></td>
<td style="text-align: left; vertical-align: top;"><code>B[64:128,0:64]</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;"><code>A[0:64,64:128]</code></td>
<td style="text-align: left; vertical-align: top;"><code>B[64:128,64:128]</code></td>
</tr>
</tbody>
</table>

`T.copy(源, 目标)`描述数据搬运。这里源是GM中的矩阵切片，目标是L1中的a或b。row、col在两轮中不变，只有ki改变，所以第二轮会读入新的K段，继续计算同一个输出块。

B按(N,K)存放，因此它的第一维对应输出列。要计算Y的第64到127列，就取B的第64到127行。`transpose_B=True`告诉`T.gemm`按`a @ b.T`计算，得到64×64的结果；不需要在GM中另建一份转置后的B。

`T.gemm(a,b,acc,...)`描述Cube矩阵乘加，结果保存在L0C的acc中。`clear_accum=(ki == 0)`决定本轮是否清除旧的累加值：

- ki=0时，条件为真，以第一轮乘积作为acc的起点。
- ki=1时，条件为假，将第二轮乘积加到已有acc上。

两轮都结束后，acc才包含完整K维的矩阵乘结果。若每轮都清除acc，最后只会保留第二轮的乘积。

从硬件上看，这段代码安排GM→L1的输入搬运，再由Cube计算，结果留在L0C。`T.gemm`内部所需的输入搬运与布局处理由后端完成。

## 5. 将结果交给Vector，加偏置后写回

先看这段程序对应的硬件路径，再逐行读向量部分。

![Matmul加Bias的代码与硬件数据路径](images/05_02_bias_hardware.png)

<small>图5-2　Matmul + Bias的数据传递与硬件分工</small>

图中的五个编号对应输入搬运、矩阵乘累加、C/V传递、偏置相加和输出写回。以右上角输出块为例，两个V核分别接收32行矩阵乘结果，并读取同一段Bias[64:128]。图中每个V核左右两份v表示同一块UB在处理前后的状态。

### 从L0C传到UB，并读取列偏置

```python
T.dual_copy(acc, v)
T.copy(Bias[col * BN:(col + 1) * BN], bias)
```

第一行将L0C中的64×64结果沿行分成两半，送到两个V核各自的UB。第二行将当前64列对应的偏置从GM搬到每个V核的局部bias中。

对于bx=1，两个V核处理的输出行分别是0～31和32～63，但列范围都是64～127，所以都读取`Bias[64:128]`。这里不能按row选偏置，偏置对应的是列。

本例的 `T.dual_copy(acc, v)` 走 **L0C→UB** 路径，中间结果没有先写入GM。图中的L0A、L0B属于`T.gemm`内部使用的输入存储，学生代码无需另写对应的搬运。

### 从UB读入寄存器，执行向量加法

这段向量加法放在 **K循环外**，等矩阵乘的全部累加完成后再执行。每个输出元素只需要加一次偏置。如果给两轮的部分乘积分别加上偏置后再求和，就会得到“矩阵乘结果＋两份偏置”，而不是所需的“矩阵乘结果＋一份偏置”。

```python
with T.SimdVF():
    full = S.pset(32, "PAT_ALL")
    for i in T.serial(BM // 2):
        value = S.vld(v[i, 0])
        bv = S.vld(bias[0])
        result = S.vadd(value, bv, full)
        S.vsts(v[i, 0], result, full)
```

`with T.SimdVF():`标出使用SIMD指令描述的向量计算区域。以下语句由Vector执行：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">语句</th>
<th style="text-align: left; vertical-align: top;">这行在做什么</th>
<th style="text-align: left; vertical-align: top;">对应的硬件操作</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>full = S.pset(32,"PAT_ALL")</code></td>
<td style="text-align: left; vertical-align: top;">创建所有通道都有效的掩码，32表示元素位宽</td>
<td style="text-align: left; vertical-align: top;">设置后续向量指令的有效通道</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>for i in T.serial(BM // 2)</code></td>
<td style="text-align: left; vertical-align: top;">遍历当前V核负责的32行，i取0到31</td>
<td style="text-align: left; vertical-align: top;">逐行组织向量处理</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>value = S.vld(v[i,0])</code></td>
<td style="text-align: left; vertical-align: top;">从第i行第0列开始，读取一组矩阵乘结果</td>
<td style="text-align: left; vertical-align: top;">UB→向量寄存器</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>bv = S.vld(bias[0])</code></td>
<td style="text-align: left; vertical-align: top;">从局部偏置的第0个元素开始，读取同样大小的一组数</td>
<td style="text-align: left; vertical-align: top;">UB→向量寄存器</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>result = S.vadd(value,bv,full)</code></td>
<td style="text-align: left; vertical-align: top;">对应位置相加，得到这一行的最终结果</td>
<td style="text-align: left; vertical-align: top;">Vector上的逐元素加法</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>S.vsts(v[i,0],result,full)</code></td>
<td style="text-align: left; vertical-align: top;">将加好偏置的一组数存回第i行</td>
<td style="text-align: left; vertical-align: top;">向量寄存器→UB</td>
</tr>
</tbody>
</table>

本例每组处理**64个float32元素，恰好是一整行**，因此只需要行循环。这里的`v[i,0]`表示向量读取的起始位置，`S.vld`会从这里读取一整组，并非只读一个标量。

每一行都使用相同的`bias[0]`作为读取起点。这样，同一列的不同行就会加上相同的偏置，完成按列广播。`S.vadd`在向量寄存器中计算，`S.vsts`将结果存回UB后，v中的内容才被更新。

> **输入块约束：** 本例固定 `BN=64`。若改为128，每行需要处理两组，并调整相应索引，不能只修改常量。

### 将两份UB结果写回GM

```python
T.dual_copy(v, Y[row * BM:(row + 1) * BM,
                 col * BN:(col + 1) * BN])
```

此时v保存的是已经加过偏置的结果。对于bx=1，这条语句将两份局部结果写入同一个64×64输出块：

- 前一份写入Y[0:32,64:128]。
- 后一份写入Y[32:64,64:128]。

源数据在两个V核各自的UB中，目标在GM中。代码给出完整输出块的切片，后端按两份局部结果对应的行安排写回。它们各写自己的32行，不会各自覆盖整个64行区域。

至此，一个结果块完成了“GM读取→Cube累加→传给Vector→加偏置→写回GM”的全过程。其他任务使用相同的代码，处理各自的输出块。

## 6. 从kernel定义到一次调用

第一小节定义了算子，实际使用时还需要准备输入、编译并调用。下面这段Host代码接在kernel定义之后，组成完整程序。

### 准备输入并调用kernel

前三份张量先在Host侧创建。A、B中的元素全为1，Bias依次为0、1、…、127。`.npu()`把输入传到设备，`.cpu()`把输出取回Host，便于检查数值。

`tilelang.compile`将前面的kernel定义编译为可调用的程序。`target="ascend"`指定昇腾后端；`out_idx=-1`指定最后一个张量参数Y是输出，由调用封装分配并返回。因此，调用`kernel(...)`时只传入A、B、Bias三个输入，返回值就是Y。

**追加Host代码**

<small>写入提示：<code>%%writefile -a</code>将下面的调用代码追加到之前保存的脚本中。</small>

In [3]:
%%writefile -a src/matmul_bias.py

import torch
import torch_npu

torch.npu.set_device(0)
a_cpu = torch.ones((M, K), dtype=torch.float16)
b_cpu = torch.ones((N, K), dtype=torch.float16)
bias_cpu = torch.arange(N, dtype=torch.float32)

kernel = tilelang.compile(matmul_bias, target="ascend", out_idx=-1)
actual = kernel(a_cpu.npu(), b_cpu.npu(), bias_cpu.npu()).cpu()
expected = a_cpu.float() @ b_cpu.float().T + bias_cpu
torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)

print(actual[0, [0, 63, 64, 127]].tolist())
print("Verification passed!")


Appending to src/matmul_bias.py


### 运行程序，查看结果

在本章目录的终端中运行，或执行下方Notebook单元：

```bash
python src/matmul_bias.py
```

In [4]:
run_result = run_and_show([sys.executable, "-u", 'src/matmul_bias.py'])


Loading tilelang libs from dev root: <已省略安装路径>
[W918 20:20:18.558302224 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 20:20:26  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `matmul_bias` with `out_idx=-1`
2026-09-18 20:20:39  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `matmul_bias`
[128.0, 191.0, 192.0, 255.0]
Verification passed!


**怎样读这组输出**

这四个数是第0行中第0、63、64、127列的结果。A、B全为1，每个完整点积都是128，四列的偏置分别为0、63、64、127，因此得到128、191、192、255。

**打印只展示了四个位置；**前面的`torch.testing.assert_close`已经将整个128×128输出与`A @ B.T + Bias`逐元素比较。只有全部比较通过后，程序才会打印成功信息。

**输入要求：** 使用本例固定的形状和类型，并保证张量连续存放。

<small>重新执行Notebook时，请从准备单元开始，依次写入kernel、追加Host代码，再运行完整程序。</small>

## 延伸阅读：C/V流水

**本例用于讲清计算与数据传递，并未验证C/V流水带来的性能收益。**

<details>
<summary>展开C/V流水的优化思路</summary>

本节沿着一个结果块讲解了完整的数据路径。处理多个独立结果块时，还可以考虑让 Vector 处理前一个块的偏置，同时让 Cube 计算后一个块的矩阵乘，以减少两类计算单元的相互等待。

实现这种重叠，需要安排好各个结果块使用的缓冲区和计算时序，保证数据准备好后才被读取，也不会在使用完之前被覆盖。仅把 Cube 和 Vector 的代码写进同一个 kernel，并不能保证形成有效的流水。

进一步优化时，需要检查实际执行是否出现重叠，并比较优化前后的耗时。

</details>

## 小结

编写这个融合算子时，我们先用`T.Tensor`约定输入输出，用`T.Kernel`选定当前结果块，再为Cube和Vector分配局部缓冲区。

计算部分由几类明确的语句组成：`T.copy`搬入输入，`T.gemm`沿K维累加，`T.dual_copy`将结果交给Vector，`S.vld → S.vadd → S.vsts`完成偏置相加，最后再由`T.dual_copy`写回输出。读代码时，可以始终追问两件事：当前数据在哪里，下一条语句是在搬运还是计算。

## 课后练习

1. bx=2时，row、col各是多少？应该写回Y的哪个区域？
2. 若把`clear_accum=(ki == 0)`改成`clear_accum=True`，本例两轮K循环结束后会保留哪些乘积？
3. 为什么两个V核的v各有32行，却都需要64个偏置？为什么读取Bias时使用col而不是row？
4. 如果去掉`S.vsts`，只保留`result = S.vadd(...)`，最后写回的还是加过偏置的结果吗？

<details>
<summary>展开练习解析</summary>

1. row=2//2=1，col=2%2=0，写入Y[64:128,0:64]。
2. 每轮都会清除旧结果，所以只保留K的后64个元素对应的乘积，丢掉第一轮贡献。
3. 两个V核各处理一半行，但都覆盖当前结果块的全部64列。偏置按列共享，因此需要相同的64个列偏置，读取范围由col决定。
4. 不是。`S.vadd`的结果在向量寄存器中；没有`S.vsts`，UB中的v仍是矩阵乘结果，最后会写回未加偏置的数据。

</details>

<small>相关文件：[Matmul + Bias完整程序](src/matmul_bias.py)。</small>

[章节导航](README.md) · [上一节](05.01_introduction.ipynb) · [下一节](05.03_dynamic_shapes.ipynb)